<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Product search for a furniture shop

**[Product search for a furniture shop](https://learning.nextia-ai.com/courses/search/m09/product-search-for-a-furniture-shop/)** · Search and Ranking: Find the Best Match · Case study 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare search designs on a real catalogue end to end (BM25, query understanding, hybrid search, a cross-encoder and LambdaMART) on quality, time and cost, and choose one for a 200 ms budget.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in about 3 minutes |
| **Needs** | An internet connection: about 130 MB of data (WANDS 96 MB, the course team's files 34 MB), and the bi-encoder (about 200 MB) in section 12. No account and no key. |
| **You should know** | The whole course; the lesson page links each lesson where it is used. |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, bm25s 0.3.16, PyStemmer 3.1.0, LightGBM 4.7.0, sentence-transformers 6.2.0 and PyTorch 2.15.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m09/product-search-for-a-furniture-shop/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M09-L01-product-search-for-a-furniture-shop/product-search-for-a-furniture-shop-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses. It needs an internet connection and takes about a minute the first time (PyTorch is about 200 MB; Colab already has it).

You should see one line per package with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 bm25s==0.3.16 PyStemmer==3.1.0 lightgbm==4.7.0 sentence-transformers==6.2.0 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'bm25s', 'PyStemmer', 'lightgbm', 'sentence-transformers', 'torch']:
    print(name, version(name))

## Setup: get the data

Run the next cell. It downloads the three WANDS files from Wayfair's repository on GitHub (about 96 MB, at a fixed version), the course team's files (about 34 MB) and the course's helper `compare.py` into a folder `wands-search`, and checks each file's checksum. It needs no account. It takes about 30 seconds.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `wands-search`, then run the cell again.

**Source and licence.** WANDS, the Wayfair ANnotation DataSet: Yan Chen, Shujian Liu, Zheng Liu, Weiyi Sun, Linas Baltrunas and Benjamin Schroeder, "WANDS: Dataset for Product Search Relevance Assessment", ECIR 2022. [github.com/wayfair/WANDS](https://github.com/wayfair/WANDS), MIT licence, Copyright (c) 2021 ecir2022. The files are not changed. The embeddings and the recorded cross-encoder scores were made from WANDS by the course team with IBM's granite models (Apache-2.0). The [dataset card](dataset.md) has the details.

In [ ]:
# Setup: download the files into wands-search/ and check each one.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

FILES = {
    "wands/product.csv": [
        "https://raw.githubusercontent.com/wayfair/WANDS/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/dataset/product.csv",
        "d993926254572e6eba96c8fd87cc549a17fb91ad3748308036eee4cf92b10ac6"
    ],
    "wands/query.csv": [
        "https://raw.githubusercontent.com/wayfair/WANDS/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/dataset/query.csv",
        "63b61660560fecc33ec490804c7e2b81402ee3e7c31a9cbb5e03736639f68e95"
    ],
    "wands/label.csv": [
        "https://raw.githubusercontent.com/wayfair/WANDS/3b74dcf4ba29ab8ff3e6a50b5b09fc627cb882b5/dataset/label.csv",
        "c11fe81ad62f17f56f316b0ec9630ebe8fbe1393578cb0ca4f05c17253a180ef"
    ],
    "data/wands_product_embeddings.npy": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/wands_product_embeddings.npy",
        "d4a3f40be0ececb53d3371483f963bdda93db6b3d8cc68c230a4af8ecab18519"
    ],
    "data/wands_query_embeddings.npy": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/wands_query_embeddings.npy",
        "85de591dcb5e713cc2fc0e9e9b1ab2ca983a35c47a7cc822f29d3f9e182544d6"
    ],
    "data/wands_rerank_scores.csv.gz": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/wands_rerank_scores.csv.gz",
        "547d0b8563c6bd73991ebe5d74e374385c8e7671f6dae60d8241202ece1c149b"
    ],
    "data/wands_latency_ms.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/wands_latency_ms.csv",
        "0fbaf7c723237da2b3713df93d46b6039f626ada279a8fc18ab05ab4a80c332b"
    ],
    "data/query_kinds.csv": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/query_kinds.csv",
        "303071929281b771c8532b345698a6e6c9ef1a68ecd35513bf1d717cfe75fd7e"
    ],
    "data/model.json": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/M09-L01-product-search-for-a-furniture-shop/data/model.json",
        "be3214d1f9e62c6fabb82cdfc8961436f453a76ebfdc5c14bbdada458c018cbb"
    ],
    "compare.py": [
        "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/compare.py",
        "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2"
    ]
}
HERE = Path.cwd()
if Path(".nextia-notebook").exists():   # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("wands-search").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:   # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, (url, expected) in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name   # a copy next to the notebook, if there is one
        shutil.copy(local, path) if local.exists() else download(url, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder wands-search and run again.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(f"Ready: {len(FILES)} files in wands-search.")

## Setup: helpers

The next cell loads PyTorch first (on macOS this avoids a clash between PyTorch and LightGBM), NumPy, pandas, matplotlib and the course helper `compare.py` as `cp`. It also defines the check helpers `expect` and `expect_hash`: they tell you if your answer is right, without showing the answer. Run it. It prints nothing.

In [ ]:
# Helpers: load PyTorch first (macOS: avoids an OpenMP clash with LightGBM), then the rest.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import torch
import json, re, time, unicodedata, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")
torch.set_num_threads(1)

def fingerprint(value):
    """A short hash of a result: the checks compare it without showing the answer."""
    import hashlib
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one of your numbers with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif isinstance(expected, float) and isinstance(yours, (int, float)) and abs(yours - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {yours}.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A furniture shop's search box gets queries such as "turquoise pillows", "chair and a half recliner" and "filaret outdoor sofa". The search team must choose a search design. It must find the best products first, and it must answer in **200 ms** (milliseconds) on a normal CPU server. In this notebook you compare five families of designs on the **full WANDS dataset**: real products, real queries and real relevance judgments from Wayfair. The [lesson page](https://learning.nextia-ai.com/courses/search/m09/product-search-for-a-furniture-shop/) tells the whole story, with the diagrams and the charts.

## 2. The data and the first checks

WANDS has three files: products, queries and judgments. A **judgment** says how well one product matches one query: Exact, Partial or Irrelevant. Run the next cell.

In [ ]:
products = pd.read_csv("wands/product.csv", sep="\t")
queries = pd.read_csv("wands/query.csv", sep="\t")
labels = pd.read_csv("wands/label.csv", sep="\t")
print(f"{len(products):,} products, {len(queries)} queries, {len(labels):,} judgments")
print(products.columns.tolist())
print(labels["label"].value_counts().to_dict())

> **Check.** 42,994 products, 480 queries and 233,448 judgments. Most judgments are Partial.

Before you measure anything, check the judgments: repeats, conflicts, and queries that cannot be scored.

In [ ]:
grades_per_pair = labels.groupby(["query_id", "product_id"])["label"].nunique()
print("repeated (query, product) pairs:", int(labels.duplicated(["query_id", "product_id"]).sum()))
print("pairs with two different labels:", int((grades_per_pair > 1).sum()))
print("products without a class:", int(products["product_class"].isna().sum()))
print("products without a description:", int(products["product_description"].isna().sum()))
exact_per_query = labels[labels["label"] == "Exact"].groupby("query_id").size()
print("queries with no exact product:", len(queries) - len(exact_per_query))
print("judged products per query: median", int(labels.groupby("query_id").size().median()))

In [ ]:
GRADE = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
first = labels.drop_duplicates(["query_id", "product_id"], keep="first")  # the first label in the file wins
judgments = {}
for q, p, g in zip(first["query_id"], first["product_id"], first["label"].map(GRADE)):
    judgments.setdefault(int(q), {})[int(p)] = int(g)
no_relevant = [q for q in queries["query_id"] if max(judgments.get(q, {0: 0}).values()) == 0]
print(f"{sum(len(j) for j in judgments.values()):,} judgments kept; "
      f"{len(no_relevant)} queries have no exact or partial product (they have no NDCG)")

> **Check.** 14 pairs have two different labels: the first label in the file wins (the course's catalogue did the same). One query has no exact or partial product, so it has no NDCG. 101 queries have no exact product, so they have no recall and no MRR.

### How this catalogue differs from Larkfield's

The course's shop, Larkfield, uses a filtered and renamed copy of WANDS. Run the next cell to see what the course left out.

In [ ]:
# The department is the first part of the category path, for example "Furniture".
products["department"] = products["category hierarchy"].fillna("").str.split(" / ").str[0].replace("", "(none)")
# The departments that Larkfield's catalogue left out (the course kept home and garden only).
DROPPED = {"School Furniture and Supplies", "Commercial Business Furniture", "Foodservice", "Protection Plans",
           "Reception Area", "Contractor", "Clips/Clamps", "Shop Product Type", "Browse By Brand"}
dropped = products["department"].isin(DROPPED)
print(f"products in departments that Larkfield left out: {int(dropped.sum())}")
print(products.loc[dropped, "department"].value_counts().to_dict())
house = products["product_name"].str.contains("wayfair", case=False)
print(f"products with the shop's own brand in the name: {int(house.sum())}")
dropped_ids = set(products.loc[dropped, "product_id"])
exact_dropped = {q for q, j in judgments.items() if any(g == 2 and p in dropped_ids for p, g in j.items())}
print(f"queries with an exact product that Larkfield left out: {len(exact_dropped)}")

Each query gets two labels for the comparison by slice: the **department** of its exact products, and its **kind**. The kinds come from a language model's recorded answers, made for the course: *broad* ("dinosaur"), *specific* ("turquoise pillows") or *known-item* (one named product, "hulmeville writing desk with hutch").

In [ ]:
# Each query gets the department of its exact products (the most common one), and its kind:
# broad, specific or known-item (a recorded answer from a language model; see the lesson).
dept_of = dict(zip(products["product_id"], products["department"]))
def query_department(q):
    j = judgments.get(q, {})
    for grade in (2, 1):
        found = [dept_of[p] for p, g in j.items() if g == grade]
        if found:
            return Counter(found).most_common(1)[0][0]
    return "(none)"
queries["department"] = queries["query_id"].map(query_department)
kinds = pd.read_csv("data/query_kinds.csv")
queries = queries.merge(kinds, on="query_id", how="left")
print(queries["department"].value_counts().head(8).to_dict())
print(queries["kind"].value_counts().to_dict())

## 3. Text into words

The search reads words. `tokens` splits the text; `norm` also removes accents, writes units one way and cuts words to their stem, as in the lesson *Normalise text*.

In [ ]:
SPLIT = re.compile(r"[^\w]+|_")
EN = __import__("Stemmer").Stemmer("english")
UNITS = [
    (re.compile(r"(\d)\s*(?:''|\"|”|″|-?inch(?:es)?\b|in\b\.?)"), r"\1 inch "),
    (re.compile(r"(\d)\s*(?:'|ft\b|feet\b|foot\b)"), r"\1 ft "),
    (re.compile(r"(\d)\s*x\s*(\d)"), r"\1 x \2"),
    (re.compile(r"(\d)\s*(cm|mm|qt|lb|lbs|gal)\b"), r"\1 \2"),
    (re.compile(r"\bx(\d)"), r"x \1"),
]

def tokens(text):
    """Plain words: lower case, split on everything that is not a letter or a digit."""
    return [t for t in SPLIT.split(str(text).lower()) if t]

def fold(text):
    """Lower case, no accents, units written one way: 'Café 24”x18' -> 'cafe 24 inch x 18'."""
    t = unicodedata.normalize("NFKD", str(text).lower())
    t = "".join(c for c in t if not unicodedata.combining(c))
    for pattern, repl in UNITS:
        t = pattern.sub(repl, t)
    return t

def norm(text):
    """Normalised and stemmed words: 'Gray Cushions 18in' -> ['gray', 'cushion', '18', 'inch']."""
    return EN.stemWords(tokens(fold(text)))

print(tokens("Gray Outdoor Cushions 18''"), norm("Gray Outdoor Cushions 18''"))

In [ ]:
KEEP = {"color": "colour", "dsprimaryproductstyle": "style", "style": "style", "primarymaterial": "material",
        "material": "material", "shape": "shape", "pattern": "pattern", "producttype": "product type"}

def attributes(features):
    """The useful attributes of WANDS's product_features ('key:value|key:value')."""
    out = {}
    for kv in str(features).split("|"):
        key, _, value = kv.partition(":")
        name = KEEP.get(key.strip())
        if name and value.strip() and name not in out:
            out[name] = value.strip()
    return " ".join(out.values())

products["title"] = products["product_name"].fillna("")
products["description"] = products["product_description"].fillna("")
products["class"] = products["product_class"].fillna("").str.replace("|", " ", regex=False)
products["category"] = products["class"] + " " + products["category hierarchy"].fillna("")
products["attributes"] = products["product_features"].map(attributes)
products["all"] = products["title"] + " " + products["description"]
IDS = products["product_id"].to_numpy()
ROW = pd.Series(np.arange(len(products)), index=products["product_id"])
print(products.loc[0, ["title", "category", "attributes"]].to_dict())

## 4. The baseline: BM25

Build the BM25 indexes: one of the title and the description with plain words (the baseline), and one per field with normalised words (for query understanding).

In [ ]:
import bm25s

def build_index(texts, analyzer):
    """A BM25 index (Lucene form, k1 1.2, b 0.75) of a list of texts."""
    index = bm25s.BM25(k1=1.2, b=0.75, method="lucene")
    index.index([analyzer(t) for t in texts], show_progress=False)
    return index

def scores(index, analyzer, text):
    """The BM25 score of every product for a query text (product order)."""
    words = [w for w in analyzer(text) if w in index.vocab_dict]
    return index.get_scores(words) if words else np.zeros(len(IDS), dtype=np.float32)

def top(score, k=100):
    """The k best product IDs with a score above 0, best first (ties by product ID)."""
    nz = np.flatnonzero(score > 0)
    if len(nz) > k:
        nz = nz[np.argpartition(-score[nz], k - 1)[:k]]
    nz = nz[np.lexsort((IDS[nz], -score[nz]))]
    return [int(IDS[i]) for i in nz]

t0 = time.time()
IDX = {"all": build_index(products["all"], tokens)}
for field in ("title", "description", "category", "attributes"):
    IDX[field] = build_index(products[field], norm)
print(f"Built 5 BM25 indexes of {len(products):,} products in {time.time() - t0:.0f} s.")

In [ ]:
RESULTS = {}

def evaluate(name, search, show=True):
    """Run search(query_id, text) on every query and keep the ranked lists and the measures."""
    run = {int(q): search(int(q), t) for q, t in zip(queries["query_id"], queries["query"])}
    pq = cp.per_query(run, judgments)
    RESULTS[name] = {"run": run, "pq": pq}
    s = cp.summary(pq)
    if show:
        print(f"{name:22s} NDCG@10 {s['ndcg@10']:.3f} ({s['ndcg@10 low']:.3f} to {s['ndcg@10 high']:.3f})  "
              f"recall@100 {s['recall@100']:.3f}  MRR {s['mrr']:.3f}")
    return pq

In [ ]:
def bm25(query_id, text, k=100):
    """The baseline: BM25 on the title and description as one field, plain words."""
    return top(scores(IDX["all"], tokens, text), k)

evaluate("bm25", bm25);

> **Check.** NDCG@10 about 0.643. That is the number to beat. Every other design must beat it on the same 480 queries, with the same judgments and the same measures (`compare.py`).

In [ ]:
title_of = dict(zip(products["product_id"], products["title"]))
def show(query_id, ranked, n=5):
    """The first n results of a query with their grade."""
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    j = judgments.get(query_id, {})
    text = queries.set_index("query_id").loc[query_id, "query"]
    print(f'"{text}"')
    for i, p in enumerate(ranked[:n], 1):
        print(f"{i}. {title_of[p][:60]:60s} [{names.get(j.get(p), 'not judged')}]")

In [ ]:
show(1, RESULTS['bm25']['run'][1])

> **Your turn.** How many queries have an **exact** product in BM25's first place? Use `RESULTS["bm25"]["run"]` (query ID → ranked product IDs) and `judgments` (query ID → {product ID: grade}, 2 = exact).

In [ ]:
exact_first = ...   # how many queries have an exact product in BM25's first place?
print(exact_first)

In [ ]:
expect('exact_first', exact_first, 223)

## 5. Query understanding

Three steps from Module 4 of the course: normalise and stem the words of each field, fix spelling, and add synonyms (added words count half). The spelling dictionary has the words of the titles, classes, categories and attributes. The synonym rules are the course's, without its planted bad rule "light, lamp".

> **Predict.** WANDS queries are typed by real people. Will spelling correction help them as much as it helped Larkfield's planted misspellings?

In [ ]:
WEIGHTS = {"title": 1.0, "description": 1.0, "category": 0.5}  # the course's validation-tuned weights

def norm_scores(text):
    """BM25 per field after normalising and stemming, added with the field weights."""
    return sum(w * scores(IDX[f], norm, text) for f, w in WEIGHTS.items())

# Spelling: a dictionary of the words in titles, classes, categories and attributes.
WORDS = Counter()
for col in ("title", "category", "attributes"):
    for t in products[col]:
        WORDS.update(tokens(fold(t)))
LETTERS = "abcdefghijklmnopqrstuvwxyz"

def edits1(w):
    splits = [(w[:i], w[i:]) for i in range(len(w) + 1)]
    return ({a + b[1:] for a, b in splits if b} | {a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1}
            | {a + c + b[1:] for a, b in splits if b for c in LETTERS} | {a + c + b for a, b in splits for c in LETTERS})

def correct_word(w):
    """A word of 4 letters or more that is not in the dictionary becomes the most frequent word 1 edit away
    (2 edits for words of 7 letters or more)."""
    if not w.isalpha() or len(w) < 4 or WORDS.get(w, 0) > 0:
        return w
    cands = [c for c in edits1(w) if c in WORDS]
    if not cands and len(w) >= 7:
        cands = [c2 for c1 in edits1(w) for c2 in edits1(c1) if c2 in WORDS]
    return max(cands, key=lambda c: (WORDS[c], c)) if cands else w

def correct(text):
    return " ".join(correct_word(w) for w in tokens(fold(text)))

# Synonyms: the course's rules (synonyms.csv) without the planted bad rule "light, lamp".
SYNONYMS = ["couch, sofa", "grey, gray", "rug, carpet", "nightstand, bedside table, night stand",
            "loveseat, love seat", "barstool, bar stool", "tv, television", "drapes, curtains",
            "comforter, duvet", "dresser, chest of drawers", "faucet, tap", "armchair, accent chair",
            "patio, outdoor", "ottoman, footstool", "throw pillow, accent pillow, decorative pillow",
            "bookcase, bookshelf", "wardrobe, armoire", "sectional => sofa", "recliner => chair"]

def expansion(text):
    """The words that the synonym rules add to a query (added words count half)."""
    q, added = f" {fold(text)} ", []
    for rule in SYNONYMS:
        if "=>" in rule:
            left, right = rule.split("=>")
            pairs = [([t.strip() for t in left.split(",")], [t.strip() for t in right.split(",")])]
        else:
            terms = [t.strip() for t in rule.split(",")]
            pairs = [([t], [o for o in terms if o != t]) for t in terms]
        for lhs, rhs in pairs:
            if any(re.search(rf"\b{re.escape(t)}\b", q) for t in lhs):
                added += [t for t in rhs if not re.search(rf"\b{re.escape(t)}\b", q)]
    return " ".join(dict.fromkeys(added))

def lexical_scores(text):
    """Query understanding: fix spelling, add synonyms (half weight), normalised BM25 per field."""
    fixed = correct(text)
    s = norm_scores(fixed)
    extra = expansion(fixed)
    return s + 0.5 * norm_scores(extra) if extra else s

evaluate("bm25-norm", lambda q, t: top(norm_scores(t)))
evaluate("bm25-norm+spell", lambda q, t: top(norm_scores(correct(t))))
evaluate("bm25-norm+spell+syn", lambda q, t: top(lexical_scores(t)));

In [ ]:
changed = [(q, t, correct(t)) for q, t in zip(queries["query_id"], queries["query"]) if correct(t) != " ".join(tokens(fold(t)))]
a, b = RESULTS["bm25-norm"]["pq"]["ndcg@10"], RESULTS["bm25-norm+spell"]["pq"]["ndcg@10"]
better = [c for c in changed if b[c[0]] > a[c[0]] + 1e-9]
worse = [c for c in changed if b[c[0]] < a[c[0]] - 1e-9]
print(f"spelling changed {len(changed)} queries: {len(better)} better, {len(worse)} worse")
for q, t, fixed in sorted(worse, key=lambda c: b[c[0]] - a[c[0]])[:4]:
    print(f"  worse: {t!r} -> {fixed!r}  NDCG@10 {a[q]:.3f} -> {b[q]:.3f}")
for q, t, fixed in sorted(better, key=lambda c: a[c[0]] - b[c[0]])[:3]:
    print(f"  better: {t!r} -> {fixed!r}  NDCG@10 {a[q]:.3f} -> {b[q]:.3f}")

> **Check.** Normalising gives the gain (0.643 → 0.706). Spelling changes few queries and hurts more than it helps (0.706 → 0.704): it "corrects" real product names, such as "filaret" → "flared". The lesson discusses what to do about it.

## 6. Dense search and hybrid search

The course team encoded every product and query once with IBM's granite bi-encoder (the files in `data/`). Dense search finds the products whose vector is closest to the query's vector. Hybrid search fuses the lexical list and the dense list with **reciprocal rank fusion** (RRF).

In [ ]:
P = np.load("data/wands_product_embeddings.npy").astype(np.float32)   # one row per product, product order
QE = np.load("data/wands_query_embeddings.npy").astype(np.float32)    # one row per query, query order
QV = dict(zip(queries["query_id"].astype(int), QE))
MODEL = json.load(open("data/model.json"))
print(f"product embeddings {P.shape}, query embeddings {QE.shape}; model {MODEL['model']}")

def dense_top(vector, k=100):
    """The k products with the highest cosine similarity (the vectors have length 1: a dot product)."""
    s = P @ vector
    idx = np.argpartition(-s, k - 1)[:k]
    idx = idx[np.lexsort((IDS[idx], -s[idx]))]
    return [int(IDS[i]) for i in idx]

evaluate("dense", lambda q, t: dense_top(QV[q]));

In [ ]:
def rrf(lists, k=60):
    """Reciprocal rank fusion: each product gets the sum of 1 / (k + rank) over the lists."""
    total = {}
    for ranked in lists:
        for rank, p in enumerate(ranked, 1):
            total[p] = total.get(p, 0.0) + 1.0 / (k + rank)
    return sorted(total.items(), key=lambda kv: (-kv[1], kv[0]))

def hybrid(query_id, text, vector=None, depth=100):
    """RRF of the top 100 of query understanding and the top 100 of dense search: [(product, rrf score)]."""
    vector = QV[query_id] if vector is None else vector
    return rrf([top(lexical_scores(text), depth), dense_top(vector, depth)])[:depth]

evaluate("hybrid-rrf", lambda q, t: [p for p, _ in hybrid(q, t)]);

> **Check.** Dense alone about 0.639, below BM25; hybrid about 0.721, the best so far.

> **Your turn (change one thing).** RRF uses k = 60. Evaluate hybrid search with k = 10, which gives the first places more weight.

In [ ]:
pq_k10 = None    # evaluate hybrid search with rrf(..., k=10) instead of k=60
k10_ndcg = ...   # its mean NDCG@10, rounded to 3 decimals
print(k10_ndcg)

In [ ]:
expect('k10_ndcg', k10_ndcg, 0.713, tolerance=0.002)

> **Check.** About 0.713: lower than with k = 60 (0.721). The course's setting also holds on the full catalogue.

## 7. Re-rank with a cross-encoder

A **cross-encoder** reads the query and one product together and gives one score. It is precise and slow. The course team scored the hybrid top 50 of every query once with IBM's granite reranker; the notebook replays those recorded scores.

> **Predict.** Which depth is best: re-rank the top 10, 20 or 50?

In [ ]:
CE = pd.read_csv("data/wands_rerank_scores.csv.gz")   # recorded: the hybrid top 50 of every query, scored once
CE_SCORE = dict(zip(zip(CE["query_id"], CE["product_id"]), CE["score"]))
print(f"{len(CE):,} recorded pairs; {CE['query_id'].nunique()} queries")

def rerank(query_id, ranked, n):
    """Re-order the first n products by the cross-encoder's score; the rest keep their order."""
    head = sorted(ranked[:n], key=lambda p: (-CE_SCORE[(query_id, p)], p))
    return head + ranked[n:]

HYB = RESULTS["hybrid-rrf"]["run"]
for n in (10, 20, 50):
    evaluate(f"rerank-ce top {n}", lambda q, t: rerank(q, HYB[q], n))

> **Check.** Top 10: 0.73; top 20: 0.746; top 50: 0.748. Every depth beats hybrid search (0.721). Keep the times in mind: the cross-encoder runs once per product.

## 8. Learning to rank with LambdaMART

LambdaMART learns how to order the hybrid top 100 from the judgments. Each candidate becomes a row of features. There is no separate training set here, so the notebook uses **cross-validation by query**: five folds of whole queries, and every query is ranked by a model that never saw it.

In [ ]:
import lightgbm as lgb

RATING = products.set_index("product_id")[["average_rating", "rating_count", "review_count"]].fillna(0)
FEATURES = ["bm25_title", "bm25_description", "bm25_category", "bm25_attributes", "bm25_plain", "dense",
            "rrf", "hybrid_rank", "lexical_rank", "dense_rank", "rating", "rating_count_log", "review_count_log",
            "query_words", "query_has_number"]

def candidates(query_id, text, vector=None, depth=100):
    """The hybrid top 100 of a query, one row per product, with the features LambdaMART reads."""
    vector = QV[query_id] if vector is None else vector
    fixed = correct(text)
    lex = top(lexical_scores(text), depth)
    den = dense_top(vector, depth)
    fused = rrf([lex, den])[:depth]
    ids = [p for p, _ in fused]
    rows = ROW[ids].to_numpy()
    f = pd.DataFrame({"product_id": ids})
    for field in ("title", "description", "category", "attributes"):
        f[f"bm25_{field}"] = scores(IDX[field], norm, fixed)[rows]
    f["bm25_plain"] = scores(IDX["all"], tokens, text)[rows]
    f["dense"] = P[rows] @ vector
    f["rrf"] = [s for _, s in fused]
    f["hybrid_rank"] = np.arange(1, len(ids) + 1)
    lex_rank, den_rank = {p: r for r, p in enumerate(lex, 1)}, {p: r for r, p in enumerate(den, 1)}
    f["lexical_rank"] = [lex_rank.get(p, depth + 1) for p in ids]
    f["dense_rank"] = [den_rank.get(p, depth + 1) for p in ids]
    r = RATING.loc[ids]
    f["rating"] = r["average_rating"].to_numpy()
    f["rating_count_log"] = np.log1p(r["rating_count"].to_numpy())
    f["review_count_log"] = np.log1p(r["review_count"].to_numpy())
    f["query_words"] = len(text.split())
    f["query_has_number"] = float(any(ch.isdigit() for ch in text))
    return f

t0 = time.time()
parts = []
for q, t in zip(queries["query_id"], queries["query"]):
    f = candidates(int(q), t)
    f["query_id"] = int(q)
    f["label"] = [judgments.get(int(q), {}).get(p, 0) for p in f["product_id"]]
    parts.append(f)
table = pd.concat(parts, ignore_index=True)
print(f"{len(table):,} candidate rows for {table['query_id'].nunique()} queries in {time.time() - t0:.0f} s; "
      f"labels {table['label'].value_counts().sort_index().to_dict()}")

In [ ]:
PARAMS = {"objective": "lambdarank", "n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31,
          "min_child_samples": 20, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
          "label_gain": [0, 1, 3], "random_state": 0, "n_jobs": 1, "verbose": -1}
# Five folds of whole queries: every query is in exactly one fold.
order = np.random.default_rng(0).permutation(queries["query_id"].to_numpy())
FOLD = {int(q): i % 5 for i, q in enumerate(order)}
table["fold"] = table["query_id"].map(FOLD)

def fit(rows, features):
    rows = rows.sort_values("query_id", kind="stable")
    model = lgb.LGBMRanker(**PARAMS)
    model.fit(rows[features], rows["label"], group=rows.groupby("query_id", sort=True).size().to_numpy())
    return model

def cross_validated(features, name):
    """Train on 4 folds, rank the 5th, five times: every query is ranked by a model that never saw it."""
    out = table[["query_id", "product_id"]].copy()
    for k in range(5):
        model = fit(table[table["fold"] != k], features)
        test = table["fold"] == k
        out.loc[test, "score"] = model.predict(table.loc[test, features])
    out = out.sort_values(["query_id", "score", "product_id"], ascending=[True, False, True])
    run = {int(q): g["product_id"].tolist() for q, g in out.groupby("query_id")}
    return evaluate(name, lambda q, t: run.get(q, []))

cross_validated(FEATURES, "ltr");

> **Check.** About 0.739: above hybrid search (0.721). LightGBM gives the same result on every run with these settings; another computer can differ in the third decimal.

> **Your turn (change one thing).** Remove the two dense features (`dense`, `dense_rank`) and measure LambdaMART again.

In [ ]:
no_dense = ...       # FEATURES without "dense" and "dense_rank"
no_dense_ndcg = ...  # the cross-validated NDCG@10 of LambdaMART with those features, rounded to 3 decimals
print(no_dense_ndcg)

In [ ]:
expect('no_dense_ndcg', no_dense_ndcg, 0.742, tolerance=0.003)

> **Check.** About 0.742, against 0.739 with them: no loss. The dense signal still reaches the model through the candidates, the RRF score and the hybrid rank. A feature can be useful to the system and useless as a column.

### A failure case: a number that is too good

Train on all 480 queries, then measure on the same queries.

In [ ]:
# A mistake on purpose: train on all 480 queries, then measure on the same queries.
model_all = fit(table, FEATURES)
seen = table[["query_id", "product_id"]].assign(score=model_all.predict(table[FEATURES]))
seen = seen.sort_values(["query_id", "score", "product_id"], ascending=[True, False, True])
seen_run = {int(q): g["product_id"].tolist() for q, g in seen.groupby("query_id")}
evaluate("ltr on its own training queries", lambda q, t: seen_run[q]);

> **Warning.** 0.882 is not a result: the model has seen the answers. Only the cross-validated 0.739 counts. The same mistake happens when you tune settings on the test queries.

One more design: the cross-encoder's score on the first 10 candidates as a feature of LambdaMART.

In [ ]:
# The cross-encoder's score as one more feature, for the first 10 hybrid candidates only (empty elsewhere).
table["ce_top10"] = [CE_SCORE.get((q, p)) if r <= 10 else np.nan
                     for q, p, r in zip(table["query_id"], table["product_id"], table["hybrid_rank"])]
cross_validated(FEATURES + ["ce_top10"], "ltr + ce top 10");
importance = pd.Series(fit(table, FEATURES).booster_.feature_importance("gain"), index=FEATURES)
print((100 * importance / importance.sum()).sort_values(ascending=False).head(6).round(1).to_dict())

## 9. One table: quality, time and cost

The time of a search is the sum of its stages. The course team measured each stage on 40 queries on its laptop (1 thread for everything but `ce…_4t`). The cost uses an assumed price of US$0.17 an hour for a 4-vCPU server.

In [ ]:
LAT = pd.read_csv("data/wands_latency_ms.csv").set_index("query_id")   # measured on the course team's laptop
BUDGET = 200   # milliseconds for one search (the budget of the course's shop)
PRICE = 0.17   # US$ an hour for a 4-vCPU server (an assumption: check your provider's price)
DESIGNS = {    # name in the table: (run, the stages it runs, threads per search)
    "bm25": ("bm25", ["bm25"], 1),
    "bm25-norm+spell+syn": ("bm25-norm+spell+syn", ["understand"], 1),
    "dense": ("dense", ["encode", "dense"], 1),
    "hybrid-rrf": ("hybrid-rrf", ["understand", "encode", "dense", "fusion"], 1),
    "rerank-ce top 10": ("rerank-ce top 10", ["understand", "encode", "dense", "fusion", "ce10"], 1),
    "rerank-ce top 10, 4 threads": ("rerank-ce top 10", ["understand", "encode", "dense", "fusion", "ce10_4t"], 4),
    "rerank-ce top 20": ("rerank-ce top 20", ["understand", "encode", "dense", "fusion", "ce20"], 1),
    "rerank-ce top 50": ("rerank-ce top 50", ["understand", "encode", "dense", "fusion", "ce50"], 1),
    "ltr": ("ltr", ["encode", "ltr"], 1),
    "ltr + ce top 10, 4 threads": ("ltr + ce top 10", ["encode", "ltr", "ce10_4t"], 4),
}
rows = []
for name, (run, stages, threads) in DESIGNS.items():
    ms = LAT[stages].sum(axis=1).to_numpy()   # the time of one search = the sum of its stages, per query
    s = cp.summary(RESULTS[run]["pq"])
    p50, p95 = cp.latency(ms)
    # A 4-vCPU server runs 4 one-thread searches at once, or one 4-thread search.
    cost = cp.cost_per_1000(p50, PRICE, threads=4 // threads)
    rows.append({"method": name, "ndcg@10": s["ndcg@10"], "low": s["ndcg@10 low"], "high": s["ndcg@10 high"],
                 "recall@100": s["recall@100"], "mrr": s["mrr"], "p50 ms": p50, "p95 ms": p95,
                 "US$ per 1,000 queries": cost})
frame = pd.DataFrame(rows)
print(frame.to_string(index=False, formatters={"p50 ms": "{:.1f}".format, "p95 ms": "{:.1f}".format,
      "US$ per 1,000 queries": "{:.5f}".format}, float_format=lambda x: f"{x:.3f}"))

In [ ]:
base = RESULTS["bm25"]["pq"]
for name in ["bm25-norm+spell+syn", "dense", "hybrid-rrf", "rerank-ce top 20", "ltr", "ltr + ce top 10"]:
    d = cp.paired(base, RESULTS[name]["pq"])
    print(f"{name:20s} {d['difference']:+.3f} ({d['low']:+.3f} to {d['high']:+.3f}); "
          f"better on {d['better']}, worse on {d['worse']}")
d = cp.paired(RESULTS["ltr"]["pq"], RESULTS["ltr + ce top 10"]["pq"])
print(f"ltr + ce top 10 against ltr: {d['difference']:+.3f} ({d['low']:+.3f} to {d['high']:+.3f})")

In [ ]:
order = ["bm25", "bm25-norm+spell+syn", "dense", "hybrid-rrf", "rerank-ce top 20", "ltr"]
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, name in enumerate(order):
    row = frame.set_index("method").loc[name]
    ax.scatter(row["p95 ms"], row["ndcg@10"], s=40)
    ax.annotate(name, (row["p95 ms"], row["ndcg@10"]), textcoords="offset points", xytext=(5, -3), fontsize=8)
ax.axvline(BUDGET, color="grey", linestyle="--"); ax.set_xscale("log")
ax.set_xlabel("p95 ms per search (log scale)"); ax.set_ylabel("NDCG@10"); ax.set_title("Quality against time")
plt.tight_layout(); plt.show()

Your computer is not the course team's laptop. Time two stages yourself and compare.

In [ ]:
# Your own computer: time the stages that need no model, on the same 40 queries.
mine = []
for q in LAT.index:
    t = queries.set_index("query_id").loc[q, "query"]
    t0 = time.perf_counter(); top(lexical_scores(t)); t1 = time.perf_counter()
    candidates(q, t); t2 = time.perf_counter()
    mine.append({"understand": 1000 * (t1 - t0), "ltr features": 1000 * (t2 - t1)})
mine = pd.DataFrame(mine)
print("your computer, p50 ms:", mine.median().round(1).to_dict())
print("course laptop, p50 ms:", {"understand": round(LAT["understand"].median(), 1), "ltr": round(LAT["ltr"].median(), 1)})

## 10. Where each design fails: slices

In [ ]:
SLICE_RUNS = ["bm25", "bm25-norm+spell+syn", "dense", "hybrid-rrf", "rerank-ce top 20", "ltr"]
def slice_table(column, min_queries=10):
    """Mean NDCG@10 of each run for each value of a query column (values with at least min_queries)."""
    cols = {}
    for name in SLICE_RUNS:
        s = cp.by_slice(RESULTS[name]["pq"], queries, column).set_index(column)
        cols[name] = s["ndcg@10"]
    t = pd.DataFrame(cols)
    t.insert(0, "queries", s["queries"])
    return t[t["queries"] >= min_queries].sort_values("queries", ascending=False)

by_kind = slice_table("kind")
by_department = slice_table("department")
print(by_kind.round(3).to_string())
print(by_department.round(3).to_string())

In [ ]:
def worst_slice(name, table):
    """The slice where a run gains least over BM25 (or loses most)."""
    gain = table[name] - table["bm25"]
    return gain.idxmin(), round(float(gain.min()), 3)
for name in SLICE_RUNS[1:]:
    print(f"{name:20s} worst kind: {worst_slice(name, by_kind)}; worst department: {worst_slice(name, by_department)}")

## 11. The decision under 200 ms

> **Your turn.** Which design has the best NDCG@10 and a p95 time within the budget? Run the next cell, then write the method's name. This is the most accurate design that fits, not yet the recommendation.

In [ ]:
fits = frame[frame["p95 ms"] <= BUDGET].sort_values("ndcg@10", ascending=False)
print(fits[["method", "ndcg@10", "p50 ms", "p95 ms", "US$ per 1,000 queries"]].to_string(index=False, formatters={
      "ndcg@10": "{:.3f}".format, "p50 ms": "{:.1f}".format, "p95 ms": "{:.1f}".format,
      "US$ per 1,000 queries": "{:.5f}".format}))

In [ ]:
choice = ...   # the method with the best NDCG@10 whose p95 time fits the budget (a string)
print(choice)

In [ ]:
expect_hash('choice', choice, '6bd7d55c1246ce1a')

The most accurate design that fits is LambdaMART with the cross-encoder on 10 candidates (4 threads). But its p95 leaves only about 22 ms for the network and the page, and it needs a whole 4-vCPU server for each search. So the [lesson page](https://learning.nextia-ai.com/courses/search/m09/product-search-for-a-furniture-shop/) recommends **hybrid search with LambdaMART** (p95 under 10 ms), and tests the cross-encoder design online against it, as the **challenger**.

## 12. End to end on new queries

Train the final LambdaMART model on all 480 queries, then search with queries that are not in WANDS. The query vector is computed now with the same bi-encoder: the first run downloads it (about 200 MB). Nobody judged these results: read them yourself.

In [ ]:
# The design for production: one LambdaMART model trained on all 480 queries (cross-validation chose the
# settings; the model that ships learns from every judgment).
final_model = fit(table, FEATURES)
encoder = None

def encode(text):
    """The query's embedding, computed now with the same bi-encoder (downloaded the first time, about 200 MB)."""
    global encoder
    if encoder is None:
        from sentence_transformers import SentenceTransformer
        encoder = SentenceTransformer(MODEL["model"], revision=MODEL["revision"], device="cpu",
                                      model_kwargs={"dtype": torch.float32})
        encoder.max_seq_length = MODEL["max_tokens"]
    return encoder.encode([text], normalize_embeddings=True, convert_to_numpy=True)[0].astype(np.float32)

def search(text, n=5):
    """The recommended design on any query text: understanding + dense -> RRF top 100 -> LambdaMART."""
    t0 = time.perf_counter()
    f = candidates(None, text, vector=encode(text))
    f["score"] = final_model.predict(f[FEATURES])
    f = f.sort_values(["score", "product_id"], ascending=[False, True])
    ms = 1000 * (time.perf_counter() - t0)
    return f["product_id"].head(n).tolist(), ms

In [ ]:
cls = dict(zip(products["product_id"], products["product_class"].fillna("").str.split("|").str[0]))
for text in ["grey outdoor cushion for a bench", "dresser for a small kids room", "walnut desk with drawers"]:
    found, ms = search(text)
    print(f'"{text}" ({ms:.0f} ms{", with loading the model" if ms > 1000 else ""})')
    for i, p in enumerate(found, 1):
        print(f"  {i}. {title_of[p][:58]:58s} {cls[p][:24]}")

> **Your turn.** Add a query of your own to the list and run the cell again. Then write down one result that a shopper would not want, and which feature could fix it.

## 13. Recap

- On all 480 real queries, BM25 scores 0.643. Normalising adds most of query understanding's gain; spelling correction hurts real product names.
- Hybrid search (0.721) beats both of its parts. A cross-encoder (0.746 on the top 20) and LambdaMART (0.739) are the best second stages.
- Only some designs fit 200 ms on a CPU. The lesson page explains the recommendation, the limits, and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Shopping queries with hard negatives](https://learning.nextia-ai.com/courses/search/m09/shopping-queries-with-hard-negatives/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m09/product-search-for-a-furniture-shop/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments are WANDS, the Wayfair ANnotation DataSet (MIT licence, Copyright (c) 2021 ecir2022), unchanged. No real customer is in this data.